qsub -N cera_run1 -v RUN_TAG=run1,NUM_SAMPLE=500000,AUTOENCODER_TYPE=CNN,ALIGNMENT_METHOD=swd,VARIABLE=pr,VAL_FRACTION=0.05,TEST_FRACTION=0.15,LAMBDA_ALIGN=0.0001,LAMBDA_PRED=0.01 run_exp5_variable_CERA_seasonal.pbs


# CMIP variable prediction — CERA-like architecture

### Data preprocessing

**Library import**

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
import torch
import torch.nn as nn

EXPERIENCE main HYPERPARAMETERS     :

In [ ]:
setup_name = "cera_seasonal"

In [ ]:
num_sample = 1000000
chosen_autoencoder_type = "CNN" # choose between "MLP" and "CNN"
inv_alignment_method = "swd"  # "swd" for sliced Wasserstein distance, "swdn" for normalized sliced Wasserstein distance, and "adversarial" for adversarial-classifier
variable = "pr"
val_fraction = 0.05
test_fraction = 0.15
cera_lambda_align = 0.0001
cera_lambda_pred = 0.01

In [ ]:
# ========================================
# CERA Hyperparameters
# ========================================

cera_n_epochs = 150

cera_align_dims = 48
cera_eval_batch_size = 2048
cera_patience = 10
cera_learning_rate = 1e-3
cera_weight_decay = 1e-3
cera_n_projections = 64

**Data Loading**

You have to run a PBS job to create the data loaded in the next cell. In the pbs file you can choose the following parameters :

- --max-abs-lat value \ (recommanded : 30)
- --patch-size-km value \ (recommanded : 1000)
- --time-stride value \ (recommanded : 24)
- --max-samples-per-climate value \ (recommanded : 1000)
- --random-seed value \ (recommanded : 42)

to run the PBS job use the following command in /glade/u/home/tsalin/CMIP: 

qsub run_build_multivariate_samples.pbs

to follow what's going on : 

qstat -u tsalin

In [ ]:
import json
from pathlib import Path

precomputed_dir = Path(f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NGS_v{num_sample}")

if not precomputed_dir.exists():
    raise FileNotFoundError(f"Precomputed data directory not found: {precomputed_dir}")

with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])
climate_colors = dict(run_cfg["climate_colors"])
selected_variables_full = list(run_cfg["selected_variables"])
max_abs_lat = float(run_cfg["max_abs_lat"])
patch_size_km = float(run_cfg["patch_size_km"])
time_stride = int(run_cfg["time_stride"])
max_samples_per_climate = int(run_cfg["max_samples_per_climate"])
random_seed = int(run_cfg["random_seed"])
n_lat = int(run_cfg["n_lat"])
n_lon = int(run_cfg["n_lon"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_patches = int(run_cfg["n_patches"])

features_by_climate_full = {
    c: np.load(precomputed_dir / f"features_{c}.npy")
    for c in climate_order
}
metadata_by_climate = {
    c: pd.read_csv(precomputed_dir / f"metadata_{c}.csv")
    for c in climate_order
}

# Split off the specified variable as a dedicated label while keeping sample/grid-point alignment.
if variable not in selected_variables_full:
    raise ValueError(f"Variable '{variable}' was not found in run_config selected_variables.")

n_variables_full = len(selected_variables_full)
expected_dim_full = n_variables_full * grid_points_per_patch
variable_var_index = selected_variables_full.index(variable)
variable_col_start = variable_var_index * grid_points_per_patch
variable_col_end = variable_col_start + grid_points_per_patch

feature_mask_without_variable = np.ones(expected_dim_full, dtype=bool)
feature_mask_without_variable[variable_col_start:variable_col_end] = False

label_variable_by_climate = {}
features_by_climate = {}
for c in climate_order:
    X_full = features_by_climate_full[c]
    if X_full.shape[1] != expected_dim_full:
        raise ValueError(
            f"Unexpected feature dimension for {c}: got {X_full.shape[1]}, expected {expected_dim_full}."
        )

    # Keep the specified variable values (one value per patch grid point) as label for each sample.
    label_variable_by_climate[c] = X_full[:, variable_col_start:variable_col_end].copy()

    # Keep all non-variable variables as model features used in the rest of the notebook.
    features_by_climate[c] = X_full[:, feature_mask_without_variable]

selected_variables = [v for v in selected_variables_full if v != variable]

# Convenience aggregate preserving same sample order as stacked climate features.
label_variable = np.vstack([label_variable_by_climate[c] for c in climate_order])

sample_count_df = pd.read_csv(precomputed_dir / "sample_count.csv").set_index("scenario")
pre_sampling_df = pd.read_csv(precomputed_dir / "pre_sampling_df.csv").set_index("scenario")
patch_catalog = pd.read_csv(precomputed_dir / "patch_catalog.csv")
sampling_diagnostics_df = pd.read_csv(precomputed_dir / "sampling_diagnostics.csv").set_index("scenario")
nan_summary_by_variable_df = pd.read_csv(precomputed_dir / "nan_summary_by_variable.csv")
sample_pairs_by_climate = {
    c: pd.read_csv(precomputed_dir / f"sample_pairs_{c}.csv")
    for c in climate_order
}

display(sample_count_df)
display(pre_sampling_df)
display(sampling_diagnostics_df)
print(f"Feature variables used downstream (without {variable}): {selected_variables}")
print(f"label_{variable} shape (all samples x grid points): {label_variable.shape}")

In [ ]:
# Backward-compatible aliases used later in the notebook
climate_order = list(climate_order)
climate_colors = dict(climate_colors)
sample_count_df = sample_count_df.copy()
pre_sampling_df = pre_sampling_df.copy()
patch_catalog = patch_catalog.copy()
sampling_diagnostics_df = sampling_diagnostics_df.copy()
nan_summary_by_variable_df = nan_summary_by_variable_df.copy()
cera_train_climates = ["historical", "ssp245"]

**Latitude Band, 1000-km Patches, and Multivariate Samples**

This step builds one unified multivariate dataset used by all downstream analyses.

Workflow:
- select data inside +/- max_abs_lat (default: 30 deg);
- split the region into non-overlapping ~1000 km x 1000 km patches;
- for each climate, build samples from all variables and all lat/lon points inside each patch at sampled times;
- split train/val/test before normalization;
- fit input and label scalers only on historical train;
- apply those scalers to all climates without refitting


Standardization

In [ ]:
# Keep explicit raw aliases for clarity. These are the unstandardized arrays loaded
# from disk after the target variable was split off.
raw_features_by_climate = {c: np.asarray(features_by_climate[c]) for c in climate_order}
raw_label_variable_by_climate = {c: np.asarray(label_variable_by_climate[c]) for c in climate_order}

print("Preprocessing will be fitted after train/val/test split using historical train only.")
print(f"Input variables: {selected_variables}")
print(f"Label variable: {variable}")


# Third Experiment - Autoencoders Trained on Increasing Climate Diversity

In this third experiment, we want to study the distribution shift of our data across climates inside an autoencoder (AE). So we focus on the latent representation of an AE trained on some climates.

We retrieve the patchs used in the second experiment. We randomly split these samples in train/val/test datasets for each climate.
We then train the AE on one of these configurations (using the train and validation sets) :
- historical climate
- historical and ssp245 climates
- historical, ssp245 and ssp370 climates
- all climates

Then we evaluate the reconstruction quality of the AE on all climates (using the test sets)

### AE construction and training


Choose the training setup at the top of this section. The AE is trained only on the train split of the climates listed in the selected setup, while all climates are kept for evaluation.


Hyperparameters of the AE

In [ ]:
ae_latent_dim = 64

Utilities :

In [ ]:
torch.manual_seed(random_seed)
np.random.seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Train/test split

In [ ]:
def build_split_indices(data_by_climate, val_fraction=0.10, test_fraction=0.20, seed=42):
    # Build train/val/test split indices for each climate.
    # The splits are done separately within each climate to ensure representation in all sets.
    split_indices = {}
    rng = np.random.default_rng(seed)

    for climate, X in data_by_climate.items():
        n = X.shape[0]
        indices = np.arange(n)
        rng.shuffle(indices)

        n_test = max(1, int(round(test_fraction * n)))
        n_val = max(1, int(round(val_fraction * n)))
        n_train = max(1, n - n_val - n_test)

        train_idx = indices[:n_train]
        val_idx = indices[n_train:n_train + n_val]
        test_idx = indices[n_train + n_val:]

        split_indices[climate] = {
            "train": train_idx,
            "val": val_idx,
            "test": test_idx,
        }

    return split_indices


# -----------------------------------------------------------------------------
# 1) Split BEFORE standardization
# -----------------------------------------------------------------------------
ae_split_indices = build_split_indices(
    raw_features_by_climate,
    val_fraction=val_fraction,
    test_fraction=test_fraction,
    seed=random_seed,
)


# ── RAM optimization: truncate eval-only climates to their test split ───────
_eval_only_climates = [c for c in climate_order if c not in cera_train_climates]
if _eval_only_climates:
    for c in _eval_only_climates:
        _idx = ae_split_indices[c]["test"]
        # raw_features_by_climate[c] and features_by_climate[c] are the same
        # numpy object (np.asarray without copy, Cell 15): update both so that
        # the 1M-row array can be freed. Same for the labels.
        _X_sub = raw_features_by_climate[c][_idx]
        raw_features_by_climate[c] = _X_sub
        features_by_climate[c]     = _X_sub
        _y_sub = raw_label_variable_by_climate[c][_idx]
        raw_label_variable_by_climate[c] = _y_sub
        label_variable_by_climate[c]     = _y_sub
        features_by_climate_full[c] = features_by_climate_full[c][_idx]
        metadata_by_climate[c] = metadata_by_climate[c].iloc[_idx].reset_index(drop=True)
        # All remaining samples are now test samples.
        ae_split_indices[c] = {
            "train": np.array([], dtype=np.intp),
            "val":   np.array([], dtype=np.intp),
            "test":  np.arange(len(_idx), dtype=np.intp),
        }
    print(f"RAM opt: {_eval_only_climates} truncated to {len(_idx)} samples (test split).")
    del _idx, _X_sub, _y_sub
del _eval_only_climates
# ── End RAM optimization ──────────────────────────────────────────────────────


# -----------------------------------------------------------------------------
# 2) Fit variable-wise scalers only on the historical train split
# -----------------------------------------------------------------------------
# The samples are flattened with a variable-major convention:
# [var0_point0, ..., var0_point69, var1_point0, ..., varN_point69].
# For the input, the target variable has already been removed. Therefore each
# input variable still occupies one contiguous block of grid_points_per_patch columns.
#
# We estimate one mean/std per physical variable using all patch grid points from
# the historical training samples. This avoids both information leakage and
# point-wise normalization that would remove local climatological structure.
reference_climate_for_scaling = "historical"

if reference_climate_for_scaling not in raw_features_by_climate:
    raise ValueError(f"Reference climate {reference_climate_for_scaling!r} not found in raw_features_by_climate.")

hist_train_idx = ae_split_indices[reference_climate_for_scaling]["train"]

X_hist_train = np.asarray(
    raw_features_by_climate[reference_climate_for_scaling][hist_train_idx],
    dtype=np.float64,
)
y_hist_train = np.asarray(
    raw_label_variable_by_climate[reference_climate_for_scaling][hist_train_idx],
    dtype=np.float64,
)

if X_hist_train.size == 0:
    raise ValueError("Historical train input set is empty; cannot fit input normalization statistics.")
if y_hist_train.size == 0:
    raise ValueError("Historical train label set is empty; cannot fit label normalization statistics.")

n_input_variables = len(selected_variables)
expected_input_dim = n_input_variables * grid_points_per_patch
if X_hist_train.shape[1] != expected_input_dim:
    raise ValueError(
        f"Unexpected input feature dimension: got {X_hist_train.shape[1]}, "
        f"expected {expected_input_dim} = {n_input_variables} variables × {grid_points_per_patch} points."
    )
if y_hist_train.shape[1] != grid_points_per_patch:
    raise ValueError(
        f"Unexpected label dimension: got {y_hist_train.shape[1]}, expected {grid_points_per_patch}."
    )

input_variable_means = np.zeros(n_input_variables, dtype=np.float64)
input_variable_stds = np.ones(n_input_variables, dtype=np.float64)

for var_idx, var_name in enumerate(selected_variables):
    cols_for_var = slice(
        var_idx * grid_points_per_patch,
        (var_idx + 1) * grid_points_per_patch,
    )
    values = X_hist_train[:, cols_for_var].reshape(-1)
    finite_values = values[np.isfinite(values)]

    if finite_values.size == 0:
        raise ValueError(f"No finite historical train values found for input variable {var_name!r}.")

    if var_name == "pr":
        finite_values = np.log1p(finite_values * 86400)

    mu = float(np.mean(finite_values))
    sigma = float(np.std(finite_values))
    if not np.isfinite(sigma) or sigma <= 0:
        sigma = 1.0

    input_variable_means[var_idx] = mu
    input_variable_stds[var_idx] = sigma

label_values = y_hist_train.reshape(-1)
label_finite_values = label_values[np.isfinite(label_values)]
if label_finite_values.size == 0:
    raise ValueError(f"No finite historical train label values found for {variable!r}.")

if variable == "pr":
    label_finite_values = np.log1p(label_finite_values * 86400)

label_variable_mean = float(np.mean(label_finite_values))
label_variable_std = float(np.std(label_finite_values))
if not np.isfinite(label_variable_std) or label_variable_std <= 0:
    label_variable_std = 1.0


def standardize_input_variablewise(X_raw):
    """Apply historical-train variable-wise normalization to the input variables."""
    X_raw = np.asarray(X_raw, dtype=np.float64)
    X_scaled = np.empty_like(X_raw, dtype=np.float32)

    for var_idx, var_name in enumerate(selected_variables):
        cols_for_var = slice(
            var_idx * grid_points_per_patch,
            (var_idx + 1) * grid_points_per_patch,
        )
        values = X_raw[:, cols_for_var]
        if var_name == "pr":
            values = np.log1p(values * 86400)
        X_scaled[:, cols_for_var] = (
            (values - input_variable_means[var_idx])
            / input_variable_stds[var_idx]
        ).astype(np.float32)

    return X_scaled


def standardize_label_variablewise(y_raw):
    """Apply historical-train target-variable normalization to all grid points of the label."""
    y_raw = np.asarray(y_raw, dtype=np.float64)
    if variable == "pr":
        y_raw = np.log1p(y_raw * 86400)
    return ((y_raw - label_variable_mean) / label_variable_std).astype(np.float32)


def denormalize_label_variable(y_scaled):
    """Convert standardized target-variable arrays back to physical units (mm/day for pr)."""
    y_scaled = np.asarray(y_scaled, dtype=np.float64)
    result = y_scaled * label_variable_std + label_variable_mean
    if variable == "pr":
        result = np.expm1(result)
    return result.astype(np.float32)


# input_scaler is intentionally not a sklearn StandardScaler because normalization is variable-wise.
class VariableWiseInputScaler:
    def __init__(self, means, stds, grid_points_per_patch):
        self.means_ = np.asarray(means, dtype=np.float64)
        self.scale_ = np.asarray(stds, dtype=np.float64)
        self.grid_points_per_patch = int(grid_points_per_patch)

    def transform(self, X):
        X = np.asarray(X, dtype=np.float64)
        out = np.empty_like(X, dtype=np.float32)
        for var_idx, var_name in enumerate(selected_variables):
            cols = slice(
                var_idx * self.grid_points_per_patch,
                (var_idx + 1) * self.grid_points_per_patch,
            )
            values = X[:, cols]
            if var_name == "pr":
                values = np.log1p(values * 86400)
            out[:, cols] = ((values - self.means_[var_idx]) / self.scale_[var_idx]).astype(np.float32)
        return out

    def inverse_transform(self, X):
        X = np.asarray(X, dtype=np.float64)
        out = np.empty_like(X, dtype=np.float32)
        for var_idx, var_name in enumerate(selected_variables):
            cols = slice(
                var_idx * self.grid_points_per_patch,
                (var_idx + 1) * self.grid_points_per_patch,
            )
            values = X[:, cols] * self.scale_[var_idx] + self.means_[var_idx]
            if var_name == "pr":
                values = np.expm1(values)
            out[:, cols] = values.astype(np.float32)
        return out


class SingleVariableLabelScaler:
    def __init__(self, mean, std):
        self.mean_ = np.array([float(mean)], dtype=np.float64)
        self.scale_ = np.array([float(std)], dtype=np.float64)

    def transform(self, y):
        y = np.asarray(y, dtype=np.float64)
        if variable == "pr":
            y = np.log1p(y * 86400)
        return ((y - self.mean_[0]) / self.scale_[0]).astype(np.float32)

    def inverse_transform(self, y):
        y = np.asarray(y, dtype=np.float64)
        result = y * self.scale_[0] + self.mean_[0]
        if variable == "pr":
            result = np.expm1(result)
        return result.astype(np.float32)


input_scaler = VariableWiseInputScaler(input_variable_means, input_variable_stds, grid_points_per_patch)
label_scaler = SingleVariableLabelScaler(label_variable_mean, label_variable_std)


# -----------------------------------------------------------------------------
# 3) Apply the historical-train variable-wise scalers to every climate without refitting
# -----------------------------------------------------------------------------
scaled_features_by_climate = {}
scaled_label_variable_by_climate = {}

for climate in climate_order:
    X_raw = np.asarray(raw_features_by_climate[climate], dtype=np.float64)
    y_raw = np.asarray(raw_label_variable_by_climate[climate], dtype=np.float64)

    scaled_features_by_climate[climate] = standardize_input_variablewise(X_raw)
    scaled_label_variable_by_climate[climate] = standardize_label_variablewise(y_raw)

# Use standardized labels in the rest of the notebook for training.
label_variable_by_climate = scaled_label_variable_by_climate
label_variable = np.vstack([label_variable_by_climate[c] for c in climate_order])

standardization_summary_df = pd.DataFrame({
    "variable": selected_variables + [variable],
    "role": ["input"] * len(selected_variables) + ["label"],
    "mean_fit_on_historical_train_all_points": list(input_variable_means) + [label_variable_mean],
    "std_fit_on_historical_train_all_points": list(input_variable_stds) + [label_variable_std],
})

print("✓ Train/val/test split created before standardization.")
print("✓ Input normalization is variable-wise: one mean/std per input variable using all grid points.")
print("✓ Label normalization is variable-wise: one mean/std for the target variable using all grid points.")
print("✓ All normalization statistics are fitted only on historical train samples.")
print("✓ The same fitted statistics are applied to all climates without refit.")
print(f"Historical train samples used for scaling: {len(hist_train_idx)}")
print(f"Input dimension after variable split: {next(iter(scaled_features_by_climate.values())).shape[1]}")
print(f"Label dimension: {next(iter(label_variable_by_climate.values())).shape[1]}")
display(standardization_summary_df)

### Seasonal encoding for the CERA predictor

In addition to the aligned latent dimensions, the predictor receives a small seasonal encoding of each sample's month, allowing it to condition its prediction on the time of year.

- **Encoding:** A 2-dimensional cyclic representation of the month,
  \[
  [\sin(2\pi\,\text{month}/12),\ \cos(2\pi\,\text{month}/12)].
  \]
  Using sine and cosine instead of the raw month index avoids the artificial discontinuity between December and January.

- **Computation:** The encoding is computed once per climate from the `month` column of the metadata, following the same sample ordering as `scaled_features_by_climate` and `label_variable_by_climate`. This also applies to evaluation-only climates, which have already been restricted to their test split during the RAM optimization step.

- **Integration into the model:** The seasonal encoding is concatenated with the aligned latent representation (`z_hist`, of size `align_dims`) before being passed to the CERA predictor, resulting in an input dimension of `align_dims + 2`.

- **Scope:** The seasonal encoding is used exclusively by the CERA prediction head. It is **not** provided to:
  - the autoencoder encoder or decoder;
  - the alignment objective (domain classifier or sliced-Wasserstein loss).

  It is only applied to the historical batch, consistently with the prediction loss, which is computed solely on historical samples.

In [ ]:
# Season encoding (sin/cos of month) given to the CERA predictor in addition to
# the aligned latent dimensions. Computed once per climate, aligned with sample
# order, so it can be indexed exactly like scaled_features_by_climate /
# label_variable_by_climate (including for eval-only climates already truncated
# to their test split by the RAM optimization above).
cera_n_season_features = 2

season_features_by_climate = {}
for climate in climate_order:
    month = metadata_by_climate[climate]["month"].to_numpy(dtype=np.float64)
    m1 = np.sin(2 * np.pi * month / 12.0)
    m2 = np.cos(2 * np.pi * month / 12.0)
    season_features_by_climate[climate] = np.stack([m1, m2], axis=1).astype(np.float32)

print(f"Season features computed for climates: {list(season_features_by_climate.keys())}")
print(f"Season feature dimension: {cera_n_season_features} (m1=sin(2*pi*month/12), m2=cos(2*pi*month/12))")


In [ ]:
# RAM Reduction 

del features_by_climate_full
del features_by_climate, raw_features_by_climate
del raw_label_variable_by_climate

Structure inspired by CERA (CNN2D) - it uses the spatial structure of the samples

In [ ]:

class CNNEncoder(nn.Module):
    def __init__(self, input_channels, spatial_shape, latent_dim):
        super().__init__()
        self.input_channels = input_channels
        self.spatial_shape = spatial_shape
        self.features = nn.Sequential(
            nn.Conv2d(input_channels, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        with torch.no_grad():
            dummy = torch.zeros(1, input_channels, *spatial_shape)
            feature_map = self.features(dummy)
            self.feature_shape = tuple(feature_map.shape[1:])
            self.flatten_dim = int(np.prod(self.feature_shape))
        self.projection = nn.Linear(self.flatten_dim, latent_dim)

    def forward(self, x):
        if x.ndim == 2:
            x = x.reshape(x.shape[0], self.input_channels, *self.spatial_shape)
        elif x.ndim != 4:
            raise ValueError("Expected a 2D flat batch or a 4D image batch.")
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.projection(x)


class CNNDecoder(nn.Module):
    def __init__(self, output_channels, spatial_shape, latent_dim, feature_shape):
        super().__init__()
        self.output_channels = output_channels
        self.spatial_shape = spatial_shape
        self.feature_shape = feature_shape
        self.project = nn.Sequential(
            nn.Linear(latent_dim, int(np.prod(feature_shape))),
            nn.ReLU(inplace=True),
        )
        self.refine = nn.Sequential(
            nn.Conv2d(feature_shape[0], 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Upsample(size=spatial_shape, mode="bilinear", align_corners=False),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, output_channels, kernel_size=3, padding=1),
        )

    def forward(self, z):
        x = self.project(z)
        x = x.reshape(z.shape[0], *self.feature_shape)
        x = self.refine(x)
        return torch.flatten(x, start_dim=1)


class CNNAutoEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim):
        super().__init__()
        self.input_channels = len(selected_variables)
        self.spatial_shape = (n_lat, n_lon)

        expected_dim = self.input_channels * grid_points_per_patch
        if input_dim != expected_dim:
            raise ValueError(
                f"input_dim={input_dim} is incompatible with a CNN reshape using "
                f"{self.input_channels} channels and {grid_points_per_patch} grid points per patch."
            )

        self.encoder = CNNEncoder(self.input_channels, self.spatial_shape, latent_dim)
        self.decoder = CNNDecoder(self.input_channels, self.spatial_shape, latent_dim, self.encoder.feature_shape)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

A more classical structure using only MLPs - it doesn't use the spatial structure but apparently it performs better

In [ ]:
input_dim = next(iter(scaled_features_by_climate.values())).shape[1]
MLP_hidden_dim_1 = max(256, min(1024, input_dim // 2))
MLP_hidden_dim_2 = max(128, min(512, input_dim // 8))


class MLPEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dims=(512, 256)):
        super().__init__()
        h1, h2 = hidden_dims
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, latent_dim),
        )

    def forward(self, x):
        return self.net(x)


class MLPDecoder(nn.Module):
    def __init__(self, latent_dim, output_dim, hidden_dims=(256, 512)):
        super().__init__()
        h1, h2 = hidden_dims
        self.net = nn.Sequential(
            nn.Linear(latent_dim, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, output_dim),
        )

    def forward(self, z):
        return self.net(z)


class MLPAutoEncoder(nn.Module):
    def __init__(self, input_dim, latent_dim, hidden_dims=(512, 256)):
        super().__init__()
        self.encoder = MLPEncoder(input_dim, latent_dim, hidden_dims=hidden_dims)
        self.decoder = MLPDecoder(latent_dim, input_dim, hidden_dims=hidden_dims[::-1])

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z


# Fourth Experiment - Invariant Autoencoder with Latent Alignment

In this fourth experiment, we dive a step closer to the real CERA architecture by adding an explicit climate invariance term to the autoencoder loss.

The idea is to test wether adding an alignment loss between climates will effectivelly bring different climate distributions closer compared to the raw data and to the simple AE architecture. Here we only train the AE on the historical climate and on SSP245 to reproduce the CERA architecture (one "cold" and one "warm" climate).

Training set:
- **historical + ssp245**

Loss:
- reconstruction loss on all samples;
- alignment loss between latent samples from **historical** and **ssp245**.

Loss equation : 
$$
L = L_{\text{rec}} + \lambda_{\text{Align}} \cdot \text{Align}(Z^{\text{hist}}_{\text{align}}, Z^{\text{ssp245}}_{\text{align}})
$$

Alignment method : 

Here we consider three different methods to align the historical and SSP245 climates ;
- **Sliced Wasserstein Distance (SWD)** (`swd`): computes the average 1D Wasserstein distance between projected latent distributions.
- **Normalized Sliced Wasserstein Distance (SWDn)** (`swdn`): same as SWD but applied after normalizing both distributions using the mean and RMS of the historical batch, i.e. SWD[(z − μ_hist) / RMS_hist]. The same normalization statistics (from hist) are applied to both historical and ssp245 latents, so that alignment focuses on distributional shape differences rather than absolute scale. Because the normalization statistics are computed per batch, a larger batch size (1024) is used for this method to ensure stable estimates.
- **Adversarial classifier** (`adversarial`): a domain classifier with gradient reversal is trained jointly to make the latent space indistinguishable between climates.

Note that in CERA, the method used is Earth Mover's Distance (EMD), but EMD can be expensive in high dimension, it is why we use a sliced Wasserstein alignment loss, which is a practical EMD-style approximation.

Note that we apply :
- reconstruction loss on all 64 latent dimensions;
- alignment applied only to the first 48 latent dimensions.

Note that the AE is either using a CNN or a simple MLP based on the setting of the Third Experiment

General architecture of the AE :

In [ ]:
def _build_autoencoder_by_type(autoencoder_type: str, input_dim: int, latent_dim: int):
    if autoencoder_type == "CNN":
        return CNNAutoEncoder(
            input_dim=input_dim,
            latent_dim=latent_dim,
        )
    if autoencoder_type == "MLP":
        return MLPAutoEncoder(
            input_dim=input_dim,
            latent_dim=latent_dim,
            hidden_dims=(MLP_hidden_dim_1, MLP_hidden_dim_2),
        )
    raise ValueError(f"Unsupported autoencoder type: {autoencoder_type}")

## Fifth Experiment - CERA-like architecture

In this fifth experiment, we add a predictor to the architecture considered in the fourth experiment. We thus now consider : AE (constructed either with cnn2D or MLPs) (and with either sliced wasserstein distance alignment or adversarial classifier alignment) and a predictor using only the aligned part of the historical climate latent representations. The predictor needs to predict the temperature field (pr) over the whole grid of the samples. This architecture will be called a CERA-like architecture.

The same test/train/val split than before is used.

We start by training the CERA-like architecture.

### Training a predictive invariant AE (historical + ssp245) - CERA like architecture

This CERA-like setup extends the invariant AE by adding a precipitation predictor:
- AE input: multivariate samples from historical + ssp245 climates.
- AE losses: weighted combination of reconstruction, latent alignment (first 48 dims), and prediction.
- Predictor: MLP on aligned latent dimensions (historical only) to predict `pr` over all 70 patch points.

Global loss (single backward pass, end-to-end training):
$$
L_{\text{total}} = (1 - \lambda_{\text{pred}} - \lambda_{\text{align}}) \cdot L_{\text{rec}} + \lambda_{\text{align}} \cdot L_{\text{align}} + \lambda_{\text{pred}} \cdot L_{\text{pred}}
$$

The reconstruction weight $(1 - \lambda_{\text{pred}} - \lambda_{\text{align}})$ ensures the three weights sum to one. AE and predictor are updated jointly in a single optimizer step.

Here are the rest of the hyperparameters

In [ ]:
# Select alignment method: "swd", "swdn" (normalized), or "adversarial".
cera_alignment_method = inv_alignment_method

# Batch size: swdn requires larger batches for stable per-batch normalization statistics.
if cera_alignment_method == "swdn":
    cera_batch_size = 1024
else:
    cera_batch_size = 1024  # Large batches help latent alignment stability.

# Predictor architecture: 5 hidden FC layers, 128 units, LeakyReLU.
cera_predictor_hidden_dim = 128
cera_predictor_n_hidden_layers = 5

# Adversarial classifier width (used only when cera_alignment_method == "adversarial").
cera_classifier_hidden_dims = (128, 64)

# Checkpoint: path to save/resume training state between PBS jobs
import re as _re

def _sanitize_ckpt(val):
    return _re.sub(r'[^A-Za-z0-9._-]+', '_', str(val).strip())

_ckpt_suffix = "_".join([
    _sanitize_ckpt(setup_name),
    f"ns{_sanitize_ckpt(num_sample)}",
    _sanitize_ckpt(chosen_autoencoder_type),
    _sanitize_ckpt(inv_alignment_method),
    _sanitize_ckpt(variable),
    _sanitize_ckpt(val_fraction),
    _sanitize_ckpt(test_fraction),
    _sanitize_ckpt(cera_lambda_align),
    _sanitize_ckpt(cera_lambda_pred),
    "seasonal",
])
cera_checkpoint_path = Path(
    f"/glade/u/home/tsalin/CMIP/model_evaluation/CERA_seasonal/cera_checkpoint_{_ckpt_suffix}.pt"
)
cera_checkpoint_freq = 1  # Save checkpoint every N epochs (1 = every epoch)

# Validate configuration
if ae_latent_dim < cera_align_dims:
    raise ValueError(f"ae_latent_dim={ae_latent_dim} must be >= cera_align_dims={cera_align_dims}")
if "label_variable_by_climate" not in globals():
    raise ValueError("label_variable_by_climate is required. Run the variable split preprocessing cell first.")

Architecture - predictor part :

In [ ]:
# CERA-like predictive invariant AE: Classes and Helper Functions

class CERAPredictor(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dim: int = 128, n_hidden_layers: int = 5):
        super().__init__()
        layers = []
        in_dim = input_dim
        for _ in range(n_hidden_layers):
            layers.append(nn.Linear(in_dim, hidden_dim))
            layers.append(nn.LeakyReLU(negative_slope=0.1))
            in_dim = hidden_dim
        layers.append(nn.Linear(in_dim, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class GradientReversalFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambd * grad_output, None


def gradient_reversal(x: torch.Tensor, lambd: float = 1.0) -> torch.Tensor:
    return GradientReversalFunction.apply(x, lambd)


class LatentDomainClassifier(nn.Module):
    def __init__(self, input_dim: int, hidden_dims=(128, 64), n_domains: int = 2):
        super().__init__()
        h1, h2 = hidden_dims
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.ReLU(),
            nn.Linear(h1, h2),
            nn.ReLU(),
            nn.Linear(h2, n_domains),
        )

    def forward(self, z: torch.Tensor, grl_lambda: float = 1.0) -> torch.Tensor:
        return self.net(gradient_reversal(z, grl_lambda))


def cera_swd_alignment_loss_torch(
    z_hist: torch.Tensor,
    z_ssp: torch.Tensor,
    n_projections: int = 64,
    eps: float = 1e-12,
) -> torch.Tensor:
    d = z_hist.shape[1]
    directions = torch.randn(n_projections, d, device=z_hist.device, dtype=z_hist.dtype)
    directions = directions / (torch.norm(directions, dim=1, keepdim=True) + eps)

    proj_hist = z_hist @ directions.T
    proj_ssp = z_ssp @ directions.T

    proj_hist_sorted, _ = torch.sort(proj_hist, dim=0)
    proj_ssp_sorted, _ = torch.sort(proj_ssp, dim=0)

    m = min(proj_hist_sorted.shape[0], proj_ssp_sorted.shape[0])
    if m == 0:
        return torch.tensor(0.0, device=z_hist.device, dtype=z_hist.dtype)

    return torch.mean(torch.abs(proj_hist_sorted[:m] - proj_ssp_sorted[:m]))

def cera_swdn_alignment_loss_torch(
    z_hist: torch.Tensor,
    z_ssp: torch.Tensor,
    n_projections: int = 64,
    eps: float = 1e-8,
) -> torch.Tensor:
    """Normalized SWD: align distributions after standardizing by historical batch stats.

    Both z_hist and z_ssp are normalized using mu and RMS computed from z_hist only,
    so the reference frame is always the historical climate of the current batch.
    SWD is then computed on the normalized latents.
    """
    mu_hist = z_hist.mean(dim=0)
    rms_hist = (z_hist ** 2).mean(dim=0).sqrt()
    z_hist_norm = (z_hist - mu_hist) / (rms_hist + eps)
    z_ssp_norm = (z_ssp - mu_hist) / (rms_hist + eps)
    return cera_swd_alignment_loss_torch(z_hist_norm, z_ssp_norm, n_projections=n_projections)



def cera_make_labeled_loader(
    data_by_climate,
    labels_by_climate,
    split_indices,
    climate: str,
    split: str,
    batch_size: int,
    shuffle: bool,
    drop_last: bool,
    season_by_climate=None,
):
    """
    Build a labeled DataLoader without unnecessary tensor copies.

    The data are already materialized as NumPy arrays. Using torch.from_numpy avoids
    an extra copy compared with torch.tensor(...). pin_memory is enabled only when a
    CUDA device is used, which can speed host-to-device transfers without changing
    the model or the data.

    When season_by_climate is provided, each batch also yields the corresponding
    sin/cos season features as a second tensor: (X, season, y) instead of (X, y).
    """
    idx = split_indices[climate][split]
    X = np.ascontiguousarray(data_by_climate[climate][idx], dtype=np.float32)
    y = np.ascontiguousarray(labels_by_climate[climate][idx], dtype=np.float32)

    if season_by_climate is not None:
        season = np.ascontiguousarray(season_by_climate[climate][idx], dtype=np.float32)
        ds = torch.utils.data.TensorDataset(
            torch.from_numpy(X),
            torch.from_numpy(season),
            torch.from_numpy(y),
        )
    else:
        ds = torch.utils.data.TensorDataset(
            torch.from_numpy(X),
            torch.from_numpy(y),
        )

    return torch.utils.data.DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        pin_memory=(device.type == "cuda"),
    )

Full training function :

In [ ]:
def train_predictive_invariant_autoencoder(
    alignment_method: str = "adversarial",
    train_climates=None,
    latent_dim: int = 64,
    n_epochs: int = 40,
    lr: float = 1e-3,
    batch_size: int = 1024,
    weight_decay: float = 1e-3,
    align_dims: int = 48,
    lambda_align: float = 0.0001,
    lambda_pred: float = 0.01,
    n_projections: int = 64,
    predictor_hidden_dim: int = 128,
    predictor_n_hidden_layers: int = 5,
    classifier_hidden_dims=(128, 64),
    checkpoint_path=None,
    checkpoint_freq=1,
 ):
    """
    Train CERA-like predictive invariant autoencoder with joint end-to-end optimization.
    
    Single backward pass and optimizer step per batch:
    L_total = (1 - lambda_pred - lambda_align) * L_rec + lambda_align * L_align + lambda_pred * L_pred
    
    - Predictor takes aligned latent dimensions concatenated with sin/cos
      season features (encoding the sample's month) as input.
    - L_rec uses reconstructions from both historical and warm batches.
    - L_align uses aligned latent parts of both batches.
    - L_pred uses predictor output and historical targets.
    - Predictor parameters are NOT frozen during AE updates.
    """
    if train_climates is None:
        train_climates = ["historical", "ssp245"]
    if train_climates != ["historical", "ssp245"]:
        raise ValueError("This setup expects train_climates=['historical', 'ssp245']")
    if alignment_method not in {"swd", "swdn", "adversarial"}:
        raise ValueError("alignment_method must be 'swd', 'swdn', or 'adversarial'.")

    input_dim_local = next(iter(scaled_features_by_climate.values())).shape[1]
    output_dim_pred = int(grid_points_per_patch)

    model = _build_autoencoder_by_type(chosen_autoencoder_type, input_dim_local, latent_dim).to(device)
    predictor = CERAPredictor(
        input_dim=align_dims + cera_n_season_features,
        output_dim=output_dim_pred,
        hidden_dim=predictor_hidden_dim,
        n_hidden_layers=predictor_n_hidden_layers,
    ).to(device)

    domain_classifier = None
    if alignment_method == "adversarial":
        domain_classifier = LatentDomainClassifier(
            input_dim=align_dims,
            hidden_dims=classifier_hidden_dims,
        ).to(device)

    # Single joint optimizer for AE + predictor + domain_classifier
    ae_and_pred_params = list(model.parameters()) + list(predictor.parameters())
    if domain_classifier is not None:
        ae_and_pred_params += list(domain_classifier.parameters())
    optimizer = torch.optim.AdamW(ae_and_pred_params, lr=lr, weight_decay=weight_decay)

    mse_loss_fn = nn.MSELoss()
    ce_loss_fn = nn.CrossEntropyLoss()

    half_batch = max(2, batch_size // 2)
    train_loader_hist = cera_make_labeled_loader(
        scaled_features_by_climate, label_variable_by_climate, ae_split_indices,
        "historical", "train", half_batch, True, True,
        season_by_climate=season_features_by_climate,
    )
    train_loader_ssp = cera_make_labeled_loader(
        scaled_features_by_climate, label_variable_by_climate, ae_split_indices,
        "ssp245", "train", half_batch, True, True
    )
    val_loader_hist = cera_make_labeled_loader(
        scaled_features_by_climate, label_variable_by_climate, ae_split_indices,
        "historical", "val", half_batch, False, True,
        season_by_climate=season_features_by_climate,
    )
    val_loader_ssp = cera_make_labeled_loader(
        scaled_features_by_climate, label_variable_by_climate, ae_split_indices,
        "ssp245", "val", half_batch, False, True
    )

    best_val = np.inf
    best_epoch = -1
    patience_counter = 0
    best_model_state = None
    best_pred_state = None
    best_domain_state = None
    history = []

    start_epoch = 1
    if checkpoint_path is not None:
        _ckpt_path = Path(checkpoint_path)
        if _ckpt_path.exists():
            _ckpt = torch.load(_ckpt_path, map_location=device)
            model.load_state_dict(_ckpt["model_state"])
            predictor.load_state_dict(_ckpt["predictor_state"])
            optimizer.load_state_dict(_ckpt["optimizer_state"])
            if domain_classifier is not None and _ckpt.get("domain_classifier_state"):
                domain_classifier.load_state_dict(_ckpt["domain_classifier_state"])
            start_epoch       = _ckpt["epoch"] + 1
            best_val          = _ckpt["best_val"]
            best_epoch        = _ckpt["best_epoch"]
            patience_counter  = _ckpt["patience_counter"]
            best_model_state  = _ckpt["best_model_state"]
            best_pred_state   = _ckpt["best_pred_state"]
            best_domain_state = _ckpt.get("best_domain_state")
            history           = _ckpt["history"]
            print(f"[CHECKPOINT] Resuming from epoch {start_epoch} "
                  f"(best: epoch {best_epoch}, val={best_val:.6f})")

    for epoch in range(start_epoch, n_epochs + 1):
        model.train()
        predictor.train()
        if domain_classifier is not None:
            domain_classifier.train()

        train_recon_losses = []
        train_align_losses = []
        train_pred_losses = []
        train_total_losses = []

        n_steps = min(len(train_loader_hist), len(train_loader_ssp))
        hist_iter = iter(train_loader_hist)
        ssp_iter = iter(train_loader_ssp)

        for _ in range(n_steps):
            xb_hist, season_hist, yb_hist = next(hist_iter)
            xb_ssp, _ = next(ssp_iter)
            xb_hist = xb_hist.to(device, non_blocking=True)
            season_hist = season_hist.to(device, non_blocking=True)
            yb_hist = yb_hist.to(device, non_blocking=True)
            xb_ssp = xb_ssp.to(device, non_blocking=True)

            # Joint end-to-end optimization: single backward pass, single optimizer step
            optimizer.zero_grad()

            # Forward pass on concatenated batch (both historical and ssp)
            xb = torch.cat([xb_hist, xb_ssp], dim=0)
            x_hat, z = model(xb)
            
            # L_rec: reconstruction loss on full concatenated batch
            recon_loss = mse_loss_fn(x_hat, xb)

            # Extract aligned latent dimensions for both climates
            z_hist = z[: xb_hist.shape[0], :align_dims]
            z_ssp = z[xb_hist.shape[0] :, :align_dims]

            # L_align: alignment loss between aligned dimensions
            if alignment_method == "swd":
                align_loss = cera_swd_alignment_loss_torch(
                    z_hist, z_ssp, n_projections=n_projections
                )
            elif alignment_method == "swdn":
                align_loss = cera_swdn_alignment_loss_torch(
                    z_hist, z_ssp, n_projections=n_projections
                )
            else:
                domain_targets = torch.cat([
                    torch.zeros(xb_hist.shape[0], dtype=torch.long, device=device),
                    torch.ones(xb_ssp.shape[0], dtype=torch.long, device=device),
                ], dim=0)
                z_align = torch.cat([z_hist, z_ssp], dim=0)
                domain_logits = domain_classifier(z_align, grl_lambda=1.0)
                align_loss = ce_loss_fn(domain_logits, domain_targets)

            # L_pred: prediction loss (only on historical batch). The predictor
            # input is the aligned latent dims concatenated with the sin/cos season
            # encoding of the sample's month.
            pred_input_hist = torch.cat([z_hist, season_hist], dim=1)
            y_pred_hist = predictor(pred_input_hist)
            pred_loss = mse_loss_fn(y_pred_hist, yb_hist)

            # Total loss: joint combination of all three losses
            total_loss = (1 - lambda_align - lambda_pred) * recon_loss + lambda_align * align_loss + lambda_pred * pred_loss
            
            # Single backward pass
            total_loss.backward()
            
            # Single optimizer step
            optimizer.step()

            train_recon_losses.append(float(recon_loss.detach().cpu().item()))
            train_align_losses.append(float(align_loss.detach().cpu().item()))
            train_pred_losses.append(float(pred_loss.detach().cpu().item()))
            train_total_losses.append(float(total_loss.detach().cpu().item()))

        model.eval()
        predictor.eval()
        if domain_classifier is not None:
            domain_classifier.eval()

        val_recon_losses = []
        val_align_losses = []
        val_pred_losses = []
        val_total_losses = []

        with torch.inference_mode():
            n_val_steps = min(len(val_loader_hist), len(val_loader_ssp))
            hist_val_iter = iter(val_loader_hist)
            ssp_val_iter = iter(val_loader_ssp)

            for _ in range(n_val_steps):
                xb_hist, season_hist, yb_hist = next(hist_val_iter)
                xb_ssp, _ = next(ssp_val_iter)
                xb_hist = xb_hist.to(device, non_blocking=True)
                season_hist = season_hist.to(device, non_blocking=True)
                yb_hist = yb_hist.to(device, non_blocking=True)
                xb_ssp = xb_ssp.to(device, non_blocking=True)

                xb = torch.cat([xb_hist, xb_ssp], dim=0)
                x_hat, z = model(xb)
                recon_loss = mse_loss_fn(x_hat, xb)

                z_hist = z[: xb_hist.shape[0], :align_dims]
                z_ssp = z[xb_hist.shape[0] :, :align_dims]

                if alignment_method == "swd":
                    align_loss = cera_swd_alignment_loss_torch(
                        z_hist, z_ssp, n_projections=n_projections
                    )
                elif alignment_method == "swdn":
                    align_loss = cera_swdn_alignment_loss_torch(
                        z_hist, z_ssp, n_projections=n_projections
                    )
                else:
                    domain_targets = torch.cat([
                        torch.zeros(xb_hist.shape[0], dtype=torch.long, device=device),
                        torch.ones(xb_ssp.shape[0], dtype=torch.long, device=device),
                    ], dim=0)
                    z_align = torch.cat([z_hist, z_ssp], dim=0)
                    domain_logits = domain_classifier(z_align, grl_lambda=1.0)
                    align_loss = ce_loss_fn(domain_logits, domain_targets)

                pred_input_hist = torch.cat([z_hist, season_hist], dim=1)
                y_pred_hist = predictor(pred_input_hist)
                pred_loss = mse_loss_fn(y_pred_hist, yb_hist)
                total_loss = (1 - lambda_align - lambda_pred) * recon_loss + lambda_align * align_loss + lambda_pred * pred_loss

                val_recon_losses.append(float(recon_loss.detach().cpu().item()))
                val_align_losses.append(float(align_loss.detach().cpu().item()))
                val_pred_losses.append(float(pred_loss.detach().cpu().item()))
                val_total_losses.append(float(total_loss.detach().cpu().item()))

        train_recon = float(np.mean(train_recon_losses)) if train_recon_losses else np.nan
        train_align = float(np.mean(train_align_losses)) if train_align_losses else np.nan
        train_pred = float(np.mean(train_pred_losses)) if train_pred_losses else np.nan
        train_total = float(np.mean(train_total_losses)) if train_total_losses else np.nan

        val_recon = float(np.mean(val_recon_losses)) if val_recon_losses else np.nan
        val_align = float(np.mean(val_align_losses)) if val_align_losses else np.nan
        val_pred = float(np.mean(val_pred_losses)) if val_pred_losses else np.nan
        val_total = float(np.mean(val_total_losses)) if val_total_losses else np.nan

        history.append(
            {
                "epoch": epoch,
                "train_recon_loss": train_recon,
                "train_align_loss": train_align,
                "train_pred_loss": train_pred,
                "train_total_loss": train_total,
                "val_recon_loss": val_recon,
                "val_align_loss": val_align,
                "val_pred_loss": val_pred,
                "val_total_loss": val_total,
            }
        )

        print(f"Epoch {epoch}/{n_epochs} | train_total={train_total:.4f} val_total={val_total:.4f} "f"(best_val={best_val:.4f} @ {best_epoch})")

        if checkpoint_path is not None and epoch % checkpoint_freq == 0:
            torch.save({
                "epoch": epoch,
                "model_state": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                "predictor_state": {k: v.cpu().clone() for k, v in predictor.state_dict().items()},
                "optimizer_state": optimizer.state_dict(),
                "domain_classifier_state": (
                    {k: v.cpu().clone() for k, v in domain_classifier.state_dict().items()}
                    if domain_classifier is not None else None
                ),
                "best_val": best_val,
                "best_epoch": best_epoch,
                "patience_counter": patience_counter,
                "best_model_state": best_model_state,
                "best_pred_state": best_pred_state,
                "best_domain_state": best_domain_state,
                "history": history,
            }, checkpoint_path)

        if val_total < best_val:
            best_val = val_total
            best_epoch = epoch
            best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_pred_state = {k: v.detach().cpu().clone() for k, v in predictor.state_dict().items()}
            if domain_classifier is not None:
                best_domain_state = {
                    k: v.detach().cpu().clone() for k, v in domain_classifier.state_dict().items()
                }
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= cera_patience:
                if checkpoint_path is not None:
                    torch.save({
                        "epoch": epoch,
                        "model_state": {k: v.cpu().clone() for k, v in model.state_dict().items()},
                        "predictor_state": {k: v.cpu().clone() for k, v in predictor.state_dict().items()},
                        "optimizer_state": optimizer.state_dict(),
                        "domain_classifier_state": (
                            {k: v.cpu().clone() for k, v in domain_classifier.state_dict().items()}
                            if domain_classifier is not None else None
                        ),
                        "best_val": best_val,
                        "best_epoch": best_epoch,
                        "patience_counter": patience_counter,
                        "best_model_state": best_model_state,
                        "best_pred_state": best_pred_state,
                        "best_domain_state": best_domain_state,
                        "history": history,
                    }, checkpoint_path)
                break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)
    if best_pred_state is not None:
        predictor.load_state_dict(best_pred_state)
    if (domain_classifier is not None) and (best_domain_state is not None):
        domain_classifier.load_state_dict(best_domain_state)

    history_df = pd.DataFrame(history)
    return model, predictor, history_df, best_epoch, domain_classifier

training :

In [ ]:
cera_ae_model, cera_predictor, cera_history_df, cera_best_epoch, cera_domain_classifier = train_predictive_invariant_autoencoder(
    alignment_method=cera_alignment_method,
    train_climates=cera_train_climates,
    latent_dim=ae_latent_dim,
    n_epochs=cera_n_epochs,
    lr=cera_learning_rate,
    batch_size=cera_batch_size,
    weight_decay=cera_weight_decay,
    align_dims=cera_align_dims,
    lambda_align=cera_lambda_align,
    lambda_pred=cera_lambda_pred,
    n_projections=cera_n_projections,
    predictor_hidden_dim=cera_predictor_hidden_dim,
    predictor_n_hidden_layers=cera_predictor_n_hidden_layers,
    classifier_hidden_dims=cera_classifier_hidden_dims,
    checkpoint_path=cera_checkpoint_path,
    checkpoint_freq=cera_checkpoint_freq,
 )

print("Chosen autoencoder type:", chosen_autoencoder_type)
print("CERA alignment method:", cera_alignment_method)
print("CERA training climates:", ", ".join(cera_train_climates))
print("Best epoch (predictive invariant AE):", cera_best_epoch)

display(cera_history_df.tail())

### Reconstruction and prediction quality on test data - EXTRACTION

We evaluate:
- AE reconstruction quality on test sets for all climates.
- the variable prediction quality from aligned latent features on test sets.

Reminder: predictor training used historical train split only.

### **Note that the denormalized precipitation is given in mm/day (instead of kg/m²/s as the input)**

In [ ]:
component_order = {"reconstruction": 0, "prediction": 1}


def _build_meta_df(component, climate, metadata, latent_dim, align_dim):
    """Build a lightweight metadata DataFrame without storing numerical arrays per row."""
    df = metadata.copy().reset_index(drop=True)
    df = df.drop(columns=["scenario"], errors="ignore")
    df.insert(0, "experiment",        "CMIP_variable_exp5_CERA")
    df.insert(1, "component",         component)
    df.insert(2, "component_order",   component_order[component])
    df.insert(3, "scenario",          climate)
    df.insert(4, "scenario_order",    int(climate_order.index(climate)))
    df.insert(5, "sample_idx",        range(len(df)))
    df.insert(6, "latent_dim",        int(latent_dim))
    df.insert(7, "align_dim",         int(align_dim))
    return df


def _predict_reconstruct_and_extract_latents_batched(
    ae_model,
    predictor,
    X,
    season,
    batch_size,
    align_dims,
):
    """
    Run one batched inference pass and return reconstruction, prediction, and latents.

    This keeps evaluation memory-efficient while preserving the same model forward pass.
    The predictor input is the aligned latent dims concatenated with the sin/cos
    season encoding of each sample's month.
    """
    X = np.ascontiguousarray(X, dtype=np.float32)
    season = np.ascontiguousarray(season, dtype=np.float32)
    ds = torch.utils.data.TensorDataset(torch.from_numpy(X), torch.from_numpy(season))
    loader = torch.utils.data.DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        drop_last=False,
        pin_memory=(device.type == "cuda"),
    )

    recon_batches = []
    pred_batches = []
    latent_batches = []

    ae_model.eval()
    predictor.eval()

    with torch.inference_mode():
        for xb_cpu, season_cpu in loader:
            xb = xb_cpu.to(device, non_blocking=True)
            season_b = season_cpu.to(device, non_blocking=True)
            x_hat, z = ae_model(xb)
            pred_input = torch.cat([z[:, :align_dims], season_b], dim=1)
            y_pred = predictor(pred_input)

            recon_batches.append(x_hat.detach().cpu().numpy())
            pred_batches.append(y_pred.detach().cpu().numpy())
            latent_batches.append(z.detach().cpu().numpy())

    return (
        np.concatenate(recon_batches, axis=0),
        np.concatenate(pred_batches, axis=0),
        np.concatenate(latent_batches, axis=0),
    )


# Names associated with flattened reconstruction and prediction vectors.
reconstruction_value_names = [
    f"{var}@point_{point_idx}"
    for var in selected_variables
    for point_idx in range(grid_points_per_patch)
]

prediction_value_names = [
    f"{variable}@point_{point_idx}"
    for point_idx in range(grid_points_per_patch)
]

meta_dfs_recon  = []
meta_dfs_pred   = []
truth_arrays_recon = []
pred_arrays_recon  = []
truth_arrays_pred  = []
pred_arrays_pred   = []

cera_latent_by_climate = {}
cera_latent_test_metadata_by_climate = {}

for climate in climate_order:
    idx = ae_split_indices[climate]["test"]
    X_scaled      = np.asarray(scaled_features_by_climate[climate][idx], dtype=np.float32)
    y_true_scaled = np.asarray(label_variable_by_climate[climate][idx],  dtype=np.float32)
    season_scaled = np.asarray(season_features_by_climate[climate][idx], dtype=np.float32)
    metadata      = metadata_by_climate[climate].iloc[idx].reset_index(drop=True)

    X_hat_scaled, y_hat_scaled, z_np = _predict_reconstruct_and_extract_latents_batched(
        ae_model=cera_ae_model,
        predictor=cera_predictor,
        X=X_scaled,
        season=season_scaled,
        batch_size=cera_eval_batch_size,
        align_dims=cera_align_dims,
    )

    cera_latent_by_climate[climate] = z_np
    cera_latent_test_metadata_by_climate[climate] = metadata

    # Reconstruction — physical units
    X_true_physical = input_scaler.inverse_transform(X_scaled)
    X_hat_physical  = input_scaler.inverse_transform(X_hat_scaled)

    meta_dfs_recon.append(_build_meta_df("reconstruction", climate, metadata, z_np.shape[1], cera_align_dims))
    truth_arrays_recon.append(X_true_physical.astype(np.float32))
    pred_arrays_recon.append(X_hat_physical.astype(np.float32))

    # Prediction — physical units
    y_true_physical = denormalize_label_variable(y_true_scaled)
    y_hat_physical  = denormalize_label_variable(y_hat_scaled)

    meta_dfs_pred.append(_build_meta_df("prediction", climate, metadata, z_np.shape[1], cera_align_dims))
    truth_arrays_pred.append(y_true_physical.astype(np.float32))
    pred_arrays_pred.append(y_hat_physical.astype(np.float32))

cera_quality_payload = {
    "meta_reconstruction":        pd.concat(meta_dfs_recon,  ignore_index=True),
    "meta_prediction":            pd.concat(meta_dfs_pred,   ignore_index=True),
    "truth_reconstruction":       np.concatenate(truth_arrays_recon, axis=0),
    "pred_reconstruction":        np.concatenate(pred_arrays_recon,  axis=0),
    "truth_prediction":           np.concatenate(truth_arrays_pred,  axis=0),
    "pred_prediction":            np.concatenate(pred_arrays_pred,   axis=0),
    "reconstruction_value_names": reconstruction_value_names,
    "prediction_value_names":     prediction_value_names,
}

print("CERA quality payload — shapes:")
print(f"  meta_reconstruction  : {cera_quality_payload['meta_reconstruction'].shape}")
print(f"  meta_prediction      : {cera_quality_payload['meta_prediction'].shape}")
print(f"  truth_reconstruction : {cera_quality_payload['truth_reconstruction'].shape}")
print(f"  pred_reconstruction  : {cera_quality_payload['pred_reconstruction'].shape}")
print(f"  truth_prediction     : {cera_quality_payload['truth_prediction'].shape}")
print(f"  pred_prediction      : {cera_quality_payload['pred_prediction'].shape}")
print()
print(cera_quality_payload["meta_reconstruction"].groupby("scenario").size().rename("n_samples_reconstruction"))
print(cera_quality_payload["meta_prediction"].groupby("scenario").size().rename("n_samples_prediction"))


### Extractiong representations of CERA-like latent space

In [ ]:
# Latent test representations were already extracted during the batched CERA evaluation cell.
# This avoids a second full pass through the autoencoder.
if "cera_latent_by_climate" not in globals() or "cera_latent_test_metadata_by_climate" not in globals():
    raise RuntimeError("Run the CERA quality/evaluation cell before this latent analysis cell.")

if cera_align_dims >= ae_latent_dim:
    raise ValueError(
        f"Cannot run non-aligned PCA because cera_align_dims={cera_align_dims} and ae_latent_dim={ae_latent_dim}."
    )


Deletion of checkpoint if the notebook ran entirely

In [ ]:
# Delete the checkpoint once the whole notebook has run successfully
if cera_checkpoint_path.exists():
    cera_checkpoint_path.unlink()
    print(f"[CHECKPOINT] Checkpoint deleted: {cera_checkpoint_path}")
else:
    print("[CHECKPOINT] No checkpoint to delete.")